In [45]:
import pandas as pd
from transformers import pipeline

In [46]:
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [47]:
result = sentiment_pipeline("ChatGPT uninstalls surged by 295% after DoD deal")
print(result)

[{'label': 'LABEL_1', 'score': 0.6729876399040222}]


In [48]:
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk
nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\nitin\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [49]:
print(sia.polarity_scores("ChatGPT uninstalls surged by 295% after DoD deal"))

{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}


In [50]:
df = pd.read_csv("reddit_features.csv")
sample = df["title"].head(5).tolist()
results = sentiment_pipeline(sample)
print(results)

[{'label': 'LABEL_1', 'score': 0.6676124930381775}, {'label': 'LABEL_1', 'score': 0.6729876399040222}, {'label': 'LABEL_1', 'score': 0.7853631377220154}, {'label': 'LABEL_1', 'score': 0.7075014114379883}, {'label': 'LABEL_0', 'score': 0.676791787147522}]


In [51]:
results = sentiment_pipeline(df["title"].tolist(), batch_size=32)
df["hf_sentiment"] = [r["label"] for r in results]
df["hf_sentiment_score"] = [r["score"] for r in results]

In [52]:
df[["title", "hf_sentiment", "hf_sentiment_score"]].head()

,title,hf_sentiment,hf_sentiment_score
0,The Removed DOGE Deposition Videos Have Alread...,LABEL_1,0.667612
1,ChatGPT uninstalls surged by 295% after DoD deal,LABEL_1,0.672987
2,1.5 Million Users Leave ChatGPT,LABEL_1,0.785363
3,‘Another internet is possible’: Norway rails a...,LABEL_1,0.707501
4,DOGE employee stole Social Security data and p...,LABEL_0,0.676792


In [53]:
df["hf_sentiment"] = df["hf_sentiment"].map({"LABEL_0": 0, "LABEL_1": 1, "LABEL_2": 2})

In [54]:
df[["title", "hf_sentiment", "hf_sentiment_score"]].head()

,title,hf_sentiment,hf_sentiment_score
0,The Removed DOGE Deposition Videos Have Alread...,1,0.667612
1,ChatGPT uninstalls surged by 295% after DoD deal,1,0.672987
2,1.5 Million Users Leave ChatGPT,1,0.785363
3,‘Another internet is possible’: Norway rails a...,1,0.707501
4,DOGE employee stole Social Security data and p...,0,0.676792


In [56]:
df.to_csv("reddit_features.csv", index=False)